In [2]:
import pandas as pd
import os

def cargarDatos():

    # Carpeta donde están los archivos
    carpeta = "estaciones"

    # Lista para guardar los DataFrames
    dfs = []

    # Recorrer todos los archivos de la carpeta
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".csv") and archivo.startswith("est_"): # Filtrar archivos CSV que comienzan con "est_"
            ruta_completa = os.path.join(carpeta, archivo)
            df = pd.read_csv(ruta_completa)
            dfs.append(df)
            print(f"Cargado: {archivo}")

    # Unir todos los DataFrames
    df_clima = pd.concat(dfs, ignore_index=True)

    print("Archivos unidos correctamente.")

    return df_clima

df_clima = cargarDatos()

Cargado: est_19.csv
Cargado: est_20.csv
Cargado: est_21.csv
Cargado: est_22.csv


C:\Users\Ismathrn\AppData\Local\Temp\ipykernel_17992\1567463792.py:16: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa)


Cargado: est_23.csv
Cargado: est_24.csv


C:\Users\Ismathrn\AppData\Local\Temp\ipykernel_17992\1567463792.py:16: DtypeWarning: Columns (2,3,4,7,8,9,10,12,13,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa)


Cargado: est_25.csv


...

Cargado: est_28.csv
Cargado: est_29.csv


C:\Users\Ismathrn\AppData\Local\Temp\ipykernel_17992\1567463792.py:16: DtypeWarning: Columns (2,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa)


Cargado: est_30.csv


C:\Users\Ismathrn\AppData\Local\Temp\ipykernel_17992\1567463792.py:16: DtypeWarning: Columns (2,3,4,7,8,9,10,12,13,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_completa)


Cargado: est_31.csv
Archivos unidos correctamente.


Se crea la columna TIMESTAMP para unir los valores de fecha y hora en un formato estandarizado que facilite el analisis posterior y asegure una Estructuración correcta de la dimensión temporal

In [3]:
df_clima["timestamp"] = pd.to_datetime(df_clima["fecha"] + " " + df_clima["hora"], errors="coerce")

porc_nat = df_clima["timestamp"].isna().mean()

print(f"Porcentaje de valores NaT en 'timestamp': {porc_nat}") 

Porcentaje de valores NaT en 'timestamp': 0.1344506654164722


In [4]:
df_clima["timestamp"].head()

0   2012-03-28 01:06:28
1   2012-03-28 01:11:28
2   2012-03-28 01:16:31
3   2012-03-28 01:46:31
4   2012-03-28 01:51:35
Name: timestamp, dtype: datetime64[ns]

In [5]:


df_clima[df_clima["timestamp"].isna()][["fecha", "hora"]].head() # Se visualizan las filas con errores


,fecha,hora
7165671,3/28/2012,23:59:59
7165672,3/28/2012,23:54:57
7165673,3/28/2012,23:49:33
7165674,3/28/2012,23:44:33
7165675,3/28/2012,23:39:31


In [6]:
df_clima[df_clima["timestamp"].isna()]["fecha"].unique()[:20]  # Se visualizan las fechas con errores

#df_clima = df_clima.sort_values(["estacion_sk", "timestamp"])

array(['3/28/2012', '3/29/2012', '3/30/2012', '3/31/2012', '4/1/2012',
       '4/2/2012', '4/3/2012', '4/4/2012', '4/25/2012', '4/26/2012',
       '4/27/2012', '4/28/2012', '4/29/2012', '4/30/2012', '5/1/2012',
       '5/2/2012', '5/3/2012', '5/4/2012', '5/5/2012', '5/6/2012'],
      dtype=object)

Aplicamos una mascara para construir un timestampt incluyendo los formatos con "/"
mask_us es una Serie True/False del mismo largo que df_clima.
df_clima.loc[filas, columnas] sirve para:
seleccionar y/o asignar valores en un subconjunto del DataFrame.

In [7]:
mask_us = df_clima["fecha"].str.contains("/")

# Para filas con formato US (con '/')
df_clima.loc[mask_us, "timestamp"] = pd.to_datetime(
    df_clima.loc[mask_us, "fecha"] + " " + df_clima.loc[mask_us, "hora"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

# Para filas con el formato "normal"
df_clima.loc[~mask_us, "timestamp"] = pd.to_datetime(
    df_clima.loc[~mask_us, "fecha"] + " " + df_clima.loc[~mask_us, "hora"],
    errors="coerce"
)

Ahora validamos nuevamente el porcentaje de error en TIMESTAMP

In [8]:
porc_nat = df_clima["timestamp"].isna().mean()

print(f"Porcentaje de valores NaT en 'timestamp': {porc_nat}") 

Porcentaje de valores NaT en 'timestamp': 0.0


Con este nuevo porcentaje de error logramos una Estructuración correcta de la dimensión temporal.
Inicialmente, el 13.44% de los timestamps no se podía interpretar por formatos mixtos; se diseñó un tratamiento diferenciado para fechas con / (%m/%d/%Y %H:%M:%S) y fechas con -, logrando 0% de NaT en timestamp.

In [ ]:
porc_nat = df_clima["timestamp"].isna().mean()

print(f"Porcentaje de valores NaT en 'timestamp': {porc_nat}") 

Porcentaje de valores NaT en 'timestamp': 0.0
